# Caldis — Getting Started

**Caldis** models energy systems (heat pumps, gas turbines, liquefiers, …) as
assemblies of components. The core idea, in one sentence:

> A component does **not** compute outputs. It declares the **equations** its
> physics must satisfy (its *residuals*), and a single solver finds the state that
> satisfies every component at once.

You never decide "who computes what": you connect components, say what is **imposed**
(pressures, temperatures, flows, design targets), and the solver does the rest.

This notebook walks through the full workflow on a small chain
**source → compressor → cooler → sink**, and shows the three things you actually do
when building a model:
1. **build & connect** the components,
2. **impose** design targets with `fix_port` (and `add_constraint`),
3. **seed** the solver — `autostart` plus, when needed, the hint methods
   (`set_start_port`, `set_start`, `set_internal_start`),
4. **solve** and **read** the result.

In [1]:
# --- imports ---
import numpy as np

import caldis
from caldis.core.system import System
from caldis.components import FluidBoundary
from caldis.components import Compressor
from caldis.components.combustion.heater import SimpleHeater   # used here as a cooler
from caldis.fluids import CoolPropBackend
from caldis.solvers import solve                 # one-call workflow (DOF -> start -> solve)
from caldis.solvers.autostart import autostart, fix_port

## 1. Build & connect

Air at 1 bar / 15 °C is compressed, then cooled back toward ambient:
- **`src`** — a `FluidBoundary` imposing the suction state (`p`, `T`) and the mass
  flow (`mdot`). Sign convention: `mdot > 0` enters a component, so a source that
  *sends* fluid out uses a negative `mdot`.
- **`comp`** — a `Compressor` (isentropic efficiency `eta_is`). It does **not** fix
  its discharge pressure — we will impose that as a design target.
- **`cooler`** — a `SimpleHeater` with `Qdot=None`: its heat duty is an **internal
  unknown**, closed by imposing the outlet temperature.
- **`sink`** — a bare `FluidBoundary` that just terminates the stream (no spec).

`connect(a, b)` wires an outlet port to the next inlet port.

In [2]:
be = CoolPropBackend("Air")

p_suction, T_suction, m = 1.0e5, 288.15, 2.0    # 1 bar, 15 C, 2 kg/s
p_discharge = 6.0e5                             # 6 bar (design target)
T_cooled    = 30 + 273.15                       # cooler outlet target

comp   = Compressor("comp", backend=be, eta_is=0.85)
cooler = SimpleHeater("cooler", backend=be, Qdot=None, dp=0)   # duty is an unknown
src    = FluidBoundary("src",  p=p_suction, T=T_suction, mdot=-m, backend=be)
sink   = FluidBoundary("sink", backend=be)                     # bare: no spec

sys_ = System("compressor_demo")
sys_.add(src, comp, cooler, sink)
sys_.connect(src.port,   comp.inl)
sys_.connect(comp.out,   cooler.inl)
sys_.connect(cooler.out, sink.port)

Connection(cooler.out <-> sink.port)

## 2. Impose design targets — `fix_port`

A model is *square* only once you have imposed as many conditions as there are free
degrees of freedom. The clean way is **`fix_port`**: it both adds the equation that
pins a port quantity **and** seeds that quantity's start, in one call — so the value
stays readable by the automatic start propagation (next step).

Here two targets:
- the compressor **discharge pressure** (`comp.out.p = 6 bar`),
- the cooler **outlet temperature** (`cooler.out.T = 30 °C`), which closes the
  cooler's unknown duty `Qdot`.

For an **arbitrary relation** between quantities (not a single port value), use
`sys_.add_constraint(lambda: ...)` instead — e.g.
`sys_.add_constraint(lambda: comp.out.p - 6*comp.inl.p, "ratio")`.

In [3]:
fix_port(sys_, comp.out,   p=p_discharge)    # discharge pressure target
fix_port(sys_, cooler.out, T=T_cooled)       # cooler outlet temperature (closes Qdot)

## 3. Seed the solver

`autostart(sys_)` propagates pressures and mass flows from the boundaries through the
component relations (equal pressures across a connection, mass conservation, a
cooler's `dp`, …). After it, most ports already have a consistent, in-domain start and
you rarely need anything else.

When a **genuinely nonlinear** quantity remains, two hint methods cover every case —
and the choice depends on whether the quantity lives on a **port** or is **internal**:

| method | scope | when |
|---|---|---|
| `set_start_port(port, p=, T=, h=, mdot=)` | one **port** quantity | the one you will use — seed a port's `p` / `T` / `h` / `mdot` |
| `set_internal_start(base_name, value)` | an **internal** unknown (no port) | the only way to seed e.g. a duty `Qdot`, a valve `Kv`, a HX `UA` |

Hints are a *starting point*, never the answer — the solver still has to satisfy every
equation. Set them **before** `autostart`.

In [4]:
# (a) set_start_port -- the one you will use: seed any PORT quantity (p / T / h / mdot).
#     The hot discharge temperature is nonlinear; a rough value helps the solver.
sys_.set_start_port(comp.out, T=500.0)

# (b) set_internal_start -- for an INTERNAL unknown that has NO port: the cooler duty Qdot.
cooler.set_internal_start("Qdot", -4.0e5)        # ~ -400 kW (cooling)
#     equivalent system-level form: sys_.set_start_internal(cooler, "Qdot", -4.0e5)

# propagate everything else (pressures, flows) from the boundaries and the fix_port anchors
autostart(sys_, verbose=True)

autostart: 12 propagated, 0 pressure fallbacks, 0 port-quantities left at default


## 4. Solve

`solve()` checks the system is square (DOF) and that the start is evaluable, then
solves. It is **Newton-first**: a direct Newton from the (autostart) seed, with
**homotopy continuation** as a fallback — you will see a `direct Newton: ok` line
when the direct attempt succeeds. After convergence it runs a **physical-validity
check** (e.g. no temperature cross) and warns, without failing, if the solution is
non-physical. On failure it prints diagnostics naming the offending equation.

In [5]:
sol = solve(sys_)

✓ DOF: square (19 unknowns)
✓ start point: evaluable
  direct Newton: ok (4 it, |F_scaled|=1.7e-10)
✓ SOLVED (|F_scaled|=1.7e-10)


## 5. Read the results

Every port exposes its solved state on demand: `port.p`, `port.mdot`, `port.h`, plus
`port.T` (K), `port.T_C` (°C), `port.x` (quality), `port.s` (entropy), recomputed
from `(p, h)`. Components expose derived outputs (here `comp.Wdot`, `cooler.Qdot`).
`system.port_table()` tabulates every port at once.

In [6]:
print(f"suction    : {comp.inl.T_C:6.1f} °C   {comp.inl.p/1e5:.2f} bar")
print(f"discharge  : {comp.out.T_C:6.1f} °C   {comp.out.p/1e5:.2f} bar")
print(f"cooled to  : {cooler.out.T_C:6.1f} °C")
print(f"mass flow  : {comp.inl.mdot:.3f} kg/s")
print(f"compressor power Wdot : {comp.Wdot/1e3:7.1f} kW  (negative = absorbed)")
print(f"cooler duty      Qdot : {cooler.Qdot/1e3:7.1f} kW  (negative = removed)")
print(f"pressure ratio        : {comp.pressure_ratio:.2f}")

sys_.port_table()

suction    :   15.0 °C   1.00 bar
discharge  :  239.3 °C   6.00 bar
cooled to  :   30.0 °C
mass flow  : 2.000 kg/s
compressor power Wdot :  -455.2 kW  (negative = absorbed)
cooler duty      Qdot :  -427.3 kW  (negative = removed)
pressure ratio        : 6.00


p_bar           T         T_C      h_kJkg   s_kJkgK    x  \
component port                                                             
src       port    1.0  288.150000   15.000000  414.377832  3.849952  2.0   
comp      inl     1.0  288.150000   15.000000  414.377832  3.849952  2.0   
          out     6.0  512.442633  239.292633  641.998550  3.918827  2.0   
cooler    inl     6.0  512.442633  239.292633  641.998550  3.918827  2.0   
          out     6.0  303.150000   30.000000  428.368744  3.383449  2.0   
sink      port    6.0  303.150000   30.000000  428.368744  3.383449  2.0   

                     rho  mdot  
component port                  
src       port  1.209506  -2.0  
comp      inl   1.209506   2.0  
          out   4.070694  -2.0  
cooler    inl   4.070694   2.0  
          out   6.906148  -2.0  
sink      port  6.906148   2.0

## 6. Explore a component

Two tools tell you what a component offers, before and after solving:
- **`structure()`** — the *names* that exist: internal variables, ports, sub-components.
- **`describe()`** — the *values* after a solve: each port's state and derived outputs.

Note the cooler's internal variable `Qdot` in its structure — that is exactly what
`set_internal_start` seeds.

In [7]:
cooler.structure()
print()
comp.describe();

SimpleHeater 'cooler'
  var    Qdot
  port   inl  (Air)   -> .p .mdot .h
  port   out  (Air)   -> .p .mdot .h
  outputs ['Qdot']

=== Compressor('comp') ===
  ports (p [bar], mdot [kg/s], T [°C]):
    inl        p=1.000  mdot=+2  h=414.4kJ/kg  T=15.00
    out        p=6.000  mdot=-2  h=642.0kJ/kg  T=239.29
  outputs:
    Wdot             = -455241
    pressure_ratio   = 6
